# R08-H59 - evidence-weighted CUSUM: the binomial log-likelihood ratio closes H50's two gaps

**Author**: Knowledge Graph Foundry autonomous build (kj) <br>
**Date**: 2026-07-07 <br>
**Pipeline stage**: R08 conformist family; decides the detector shipped by H120 into the SOTA config <br>
**Data**: wave-1b realized drift series (474 post-cure docs, single spike at doc 75), H50 protocol <br>

H50 left CUSUM the class winner with two measured gaps: the plain-rate form false-alarmed on the
real stream's single small-denominator spike, and its threshold did not transfer across noise
floors. The LLR form scores each document by evidence weight:

$$S_i = \max\left(0,\; S_{i-1} + \log\frac{\mathrm{Bin}(k_i;\, n_i,\, p_1)}{\mathrm{Bin}(k_i;\, n_i,\, \hat p_0)}\right)$$

A rate of 1.0 on a 3-entity document carries little log-likelihood; the same rate on a 30-entity
document carries a lot. p0-hat adapts quiescently (EWMA only while S == 0, per H50's fix); p1 =
p0-hat + delta_design.

## Pre-registered clauses (all three must hold)
1. Zero alarms on the realized wave-1b series including the spike document
2. Threshold h calibrated at p0 = 0.05 lands in the ARL0 band (420-600) at p0 = 0.01 and 0.10 without recalibration
3. Detection miss/delay matches or beats plain quiescent CUSUM at matched ARL0 on step and ramp shifts

## Outputs
- `reports/drift-llr-h59-<stamp>.json` - calibration, cross-floor ARL0, races, real-series check

In [1]:
# Imports
import datetime, json
import numpy as np
from pathlib import Path
from scipy.stats import binom
from rich import print as rprint
from rich.progress import Progress

rng = np.random.default_rng(42)

# H50 protocol constants
ARL_BAND = (420, 600)
CAL_LEN = 2500
N_CAL_STREAMS = 60
ONSET, HORIZON = 300, 40
N_TRIALS = 200
FLOORS = [0.01, 0.05, 0.10]
DELTA = 0.30           # tested shift (H50)
DELTA_DESIGN = 0.30    # p1 = p0_hat + DELTA_DESIGN in the LLR
EWMA = 0.05            # quiescent adaptation rate
CUSUM_ALLOWANCE = 0.05 # plain-CUSUM slack (H50 winner's config)

series = json.load(open("../reports/wave1b-drift-series.json"))
REAL_RATES = np.array(series["remap_rates"])
REAL_N = np.array(series["entities_touched_per_doc"][-len(REAL_RATES):])
DOC_COUNTS = REAL_N.copy()  # realized entities-per-doc distribution for simulation

rprint(f"real series: [yellow]{len(REAL_RATES)}[/yellow] docs, spike at doc "
       f"[yellow]{int(np.argmax(REAL_RATES))}[/yellow] (rate {REAL_RATES.max():.2f}, "
       f"n={REAL_N[int(np.argmax(REAL_RATES))]})")

real series: 474 docs, spike at doc 75 (rate 1.00, n=3)

In [2]:
def sample_n(size):
    return rng.choice(DOC_COUNTS, size=size)

def simulate_stream(p_path):
    n = sample_n(len(p_path))
    k = rng.binomial(n, p_path)
    return k, n

def alarms_llr(k, n, h):
    s, mu0, out = 0.0, None, []
    for i in range(len(k)):
        xi = k[i] / max(n[i], 1)
        if mu0 is None:
            mu0 = max(xi, 1e-3)
        if s == 0.0:
            mu0 = (1 - EWMA) * mu0 + EWMA * xi
        p0 = np.clip(mu0, 1e-3, 0.5)
        p1 = np.clip(p0 + DELTA_DESIGN, p0 + 1e-3, 0.999)
        llr = binom.logpmf(k[i], n[i], p1) - binom.logpmf(k[i], n[i], p0)
        s = max(0.0, s + llr)
        if s >= h:
            out.append(i); s = 0.0
    return out

def alarms_plain(k, n, h):
    s, mu0, out = 0.0, None, []
    for i in range(len(k)):
        xi = k[i] / max(n[i], 1)
        if mu0 is None:
            mu0 = xi
        if s == 0.0:
            mu0 = (1 - EWMA) * mu0 + EWMA * xi
        s = max(0.0, s + xi - (mu0 + CUSUM_ALLOWANCE))
        if s >= h:
            out.append(i); s = 0.0
    return out

def arl0(fn, h, p0, n_streams=N_CAL_STREAMS):
    gaps = []
    for _ in range(n_streams):
        k, n = simulate_stream(np.full(CAL_LEN, p0))
        a = fn(k, n, h)
        gaps.append(a[0] + 1 if a else CAL_LEN + 1)
    return float(np.mean(gaps))

def calibrate(fn, p0, lo, hi):
    for _ in range(18):
        mid = (lo + hi) / 2
        a = arl0(fn, mid, p0)
        if a < ARL_BAND[0]:
            lo = mid
        elif a > ARL_BAND[1]:
            hi = mid
        else:
            return mid, a, True
    return mid, a, ARL_BAND[0] <= a <= ARL_BAND[1]

h_llr, a_llr, ok1 = calibrate(alarms_llr, 0.05, 0.5, 60.0)
h_pln, a_pln, ok2 = calibrate(alarms_plain, 0.05, 0.05, 8.0)
rprint(f"calibrated at p0=0.05: LLR h=[yellow]{h_llr:.2f}[/yellow] (ARL0 {a_llr:.0f}, in-band {ok1})  "
       f"plain h=[yellow]{h_pln:.3f}[/yellow] (ARL0 {a_pln:.0f}, in-band {ok2})")

calibrated at p0=0.05: LLR h=3.75 (ARL0 486, in-band True)  plain h=0.267 (ARL0 465, in-band True)

In [3]:
# Clause 2 - cross-floor ARL0 transfer without recalibration
transfer = {}
for p0 in FLOORS:
    transfer[p0] = {"llr": arl0(alarms_llr, h_llr, p0), "plain": arl0(alarms_plain, h_pln, p0)}
clause2 = all(ARL_BAND[0] <= transfer[p0]["llr"] <= ARL_BAND[1] for p0 in [0.01, 0.10])
rprint("[bold cyan]Cross-floor ARL0 (thresholds fixed from 0.05)[/bold cyan]")
for p0, r in transfer.items():
    rprint(f"  p0={p0}: LLR [yellow]{r['llr']:.0f}[/yellow]  plain [yellow]{r['plain']:.0f}[/yellow]  [dim]band {ARL_BAND}[/dim]")
rprint(f"  clause 2 (LLR in band at 0.01 and 0.10): [{'green' if clause2 else 'red'}]{clause2}[/]")

Cross-floor ARL0 (thresholds fixed from 0.05)

p0=0.01: LLR 1908  plain 2343  band (420, 600)

p0=0.05: LLR 607  plain 362  band (420, 600)

p0=0.1: LLR 525  plain 112  band (420, 600)

clause 2 (LLR in band at 0.01 and 0.10): False

In [4]:
# Clause 3 - detection races at matched budget (per-floor recalibrated thresholds for fairness)
def race(fn, h, p0, shape):
    miss, delays = 0, []
    for _ in range(N_TRIALS):
        p = np.full(ONSET + HORIZON + 200, p0)
        if shape == "step":
            p[ONSET:] = min(p0 + DELTA, 0.99)
        else:
            ramp = np.linspace(p0, min(p0 + DELTA, 0.99), 10)
            p[ONSET:ONSET + 10] = ramp
            p[ONSET + 10:] = min(p0 + DELTA, 0.99)
        k, n = simulate_stream(p)
        a = [x for x in fn(k, n, h) if ONSET <= x <= ONSET + HORIZON]
        if a:
            delays.append(a[0] - ONSET)
        else:
            miss += 1
    return miss / N_TRIALS, (float(np.median(delays)) if delays else None)

races = {}
with Progress() as prog:
    t = prog.add_task("races", total=len(FLOORS) * 4)
    for p0 in FLOORS:
        hl, _, _ = calibrate(alarms_llr, p0, 0.5, 60.0)
        hp, _, _ = calibrate(alarms_plain, p0, 0.05, 8.0)
        for shape in ("step", "ramp"):
            races[f"{p0}-{shape}"] = {
                "llr": race(alarms_llr, hl, p0, shape),
                "plain": race(alarms_plain, hp, p0, shape)}
            prog.advance(t, 2)
clause3 = all(races[k]["llr"][0] <= races[k]["plain"][0] for k in races)
rprint("[bold cyan]Detection races (miss rate, median delay)[/bold cyan]")
for k, r in races.items():
    rprint(f"  {k:10s} LLR miss [yellow]{r['llr'][0]:.2f}[/yellow] delay {r['llr'][1]}  |  "
           f"plain miss [yellow]{r['plain'][0]:.2f}[/yellow] delay {r['plain'][1]}")
rprint(f"  clause 3 (LLR miss <= plain everywhere): [{'green' if clause3 else 'red'}]{clause3}[/]")

/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Detection races (miss rate, median delay)

0.01-step  LLR miss 0.00 delay 0.0  |  plain miss 0.00 delay 0.0

0.01-ramp  LLR miss 0.00 delay 4.0  |  plain miss 0.00 delay 4.0

0.05-step  LLR miss 0.00 delay 0.0  |  plain miss 0.00 delay 1.0

0.05-ramp  LLR miss 0.00 delay 6.0  |  plain miss 0.00 delay 5.0

0.1-step   LLR miss 0.00 delay 1.0  |  plain miss 0.00 delay 1.0

0.1-ramp   LLR miss 0.00 delay 6.0  |  plain miss 0.00 delay 6.0

clause 3 (LLR miss <= plain everywhere): True

In [5]:
# Clause 1 - the realized wave-1b series, spike included
k_real = np.round(REAL_RATES * REAL_N).astype(int)
alarms_real_llr = alarms_llr(k_real, REAL_N, h_llr)
alarms_real_plain = alarms_plain(k_real, REAL_N, h_pln)
clause1 = len(alarms_real_llr) == 0

verdict = "CONFIRMED" if clause1 and clause2 and clause3 else "REFUTED"
rprint(f"""[bold cyan]H59 verdict[/bold cyan]
[dim]{"\u2500" * 40}[/dim]
  Clause 1 - real series alarms: LLR [{'green' if clause1 else 'red'}]{alarms_real_llr}[/]  plain [yellow]{alarms_real_plain}[/yellow]
  Clause 2 - cross-floor transfer: [{'green' if clause2 else 'red'}]{clause2}[/]
  Clause 3 - matched-budget detection: [{'green' if clause3 else 'red'}]{clause3}[/]
  Verdict: [{'green' if verdict == 'CONFIRMED' else 'red'}]{verdict}[/]
""")

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
out = Path("../reports") / f"drift-llr-h59-{stamp}.json"
out.write_text(json.dumps({
    "h_llr": h_llr, "h_plain": h_pln, "arl0_cal": {"llr": a_llr, "plain": a_pln},
    "cross_floor": transfer, "races": races,
    "real_series": {"llr_alarms": alarms_real_llr, "plain_alarms": alarms_real_plain},
    "clauses": [bool(clause1), bool(clause2), bool(clause3)], "verdict": verdict,
}, indent=2, default=str))
rprint("saved", str(out))

H59 verdict
────────────────────────────────────────
  Clause 1 - real series alarms: LLR [75]  plain [75]
  Clause 2 - cross-floor transfer: False
  Clause 3 - matched-budget detection: True
  Verdict: REFUTED

saved ../reports/drift-llr-h59-20260707-090229.json